# GPU Compute Cost Analysis — FATURA Dataset Annotation

## 1. Compute Requirement

The current objective is to create a high-quality training dataset from the **10,000 documents of the FATURA dataset**.

The existing FATURA annotations cannot be directly used for fine-tuning because they do not match the desired output structure and contain several limitations:

* some fields are too generic and need to be split into more specific fields;
* some required fields are missing;
* line-item information is not sufficiently represented;
* field presence is inconsistent between documents;
* some annotations contain errors;
* the existing annotations therefore cannot be considered a complete representation of the information available in the original documents.

A simple schema-conversion process would only solve the formatting and label-specialisation problem. It cannot recover information that is absent from the original annotations.

The chosen approach is therefore to **re-annotate the 10,000 original document images directly using a large Vision-Language Model (VLM)**.

The resulting dataset will then be used to fine-tune a **much smaller VLM** that can be deployed for production.

# 2. Why a Large VLM Is Required for Annotation

The initial experiment used **Qwen2.5-VL-7B-Instruct** to annotate 200 randomly selected documents.

The results were promising in terms of the model's ability to understand the documents, but the extraction quality was not sufficiently reliable to use the generated annotations directly as a 10,000-document training dataset.

Since the teacher model has **not been fine-tuned specifically for our extraction task**, increasing the model capacity is the next approach to evaluate.

A larger model is expected to provide stronger:

* document understanding;
* reasoning over complex layouts;
* field identification;
* association between values and their semantic meaning;
* handling of different invoice templates;
* extraction of structured information.

The objective is not to deploy this large model in production.

Its role is to act as a **teacher model for data generation**.

Once the 10,000 documents have been annotated, the resulting dataset can be used to fine-tune a significantly smaller VLM specifically for our document types and target JSON structure.

This creates a trade-off:

> **Higher compute cost during dataset generation → lower model size and inference cost in production.**

# 3. GPU Memory Requirement

The main infrastructure constraint is GPU VRAM.

The next experiments require models substantially larger than the 7B model initially tested. A key candidate is a **32B-class VLM**, such as Qwen2.5-VL-32B.

A 32B-parameter model stored in BF16/FP16 requires approximately:

$$
32B \times 2\ bytes \approx 64GB
$$

of memory for the model weights alone.

Additional GPU memory is required for:

* KV cache;
* activations;
* image processing;
* intermediate tensors;
* framework overhead;
* generation.

Consequently, a GPU with **80 GB VRAM** provides an appropriate baseline for evaluating a 32B-class VLM in higher precision.

Quantization can reduce the memory requirement, but the objective of this phase is to benchmark a large teacher model reliably rather than optimize the teacher for deployment.

For this reason, GPUs below 80 GB are excluded from the main comparison.

The relevant GPU class includes:

* NVIDIA A100 80 GB
* NVIDIA H100 80 GB
* NVIDIA H100 NVL 94 GB
* NVIDIA H200 141 GB
* other GPUs with at least approximately 80 GB VRAM

The **H100 80 GB** is used as the main reference GPU because it is widely available across cloud providers and provides sufficient memory for the planned experiments.

# 4. Types of Compute Providers

GPU rental options can be divided into three main categories.

## 4.1 Hyperscale Cloud Providers

Examples:

* AWS
* Microsoft Azure
* Google Cloud

These providers offer GPU instances as part of large general-purpose cloud ecosystems.

They provide:

* virtual machines;
* GPU computing;
* storage;
* networking;
* identity and security;
* monitoring;
* Kubernetes;
* databases;
* enterprise support;
* multi-region infrastructure.

### Advantages

* mature and reliable infrastructure;
* strong security and enterprise capabilities;
* integration with existing cloud services;
* large-scale deployment capabilities;
* suitable for production environments.

### Disadvantages

* higher GPU prices;
* more complex pricing;
* additional infrastructure costs can apply;
* generally more infrastructure than is required for a temporary annotation workload.

For the current dataset-generation phase, most of these additional services are not essential.

The requirement is primarily:

> **One high-VRAM GPU capable of running the teacher model.**

## 4.2 AI-Specialized Cloud Providers

Examples:

* Verda
* Lambda
* CoreWeave
* Nebius
* Crusoe
* Scaleway

These providers focus heavily on GPU and AI workloads.

Their infrastructure still provides complete compute environments, but the service is optimized around GPU workloads rather than a broad general-purpose cloud ecosystem.

### Advantages

* lower GPU prices;
* high-end GPU availability;
* simple deployment;
* optimized AI infrastructure;
* suitable for training and inference.

### Disadvantages

* smaller general-purpose cloud ecosystem;
* fewer enterprise services than AWS/Azure/GCP;
* potentially fewer regions and integrations.

For the current annotation workload, this category is highly relevant because the project primarily requires GPU compute.

## 4.3 Developer GPU Clouds / GPU Marketplaces

Examples:

* RunPod
* Paperspace

These platforms focus on providing relatively simple access to GPUs.

They are particularly suited to:

* experimentation;
* benchmarking;
* fine-tuning;
* batch inference;
* dataset generation.

### Advantages

* simple deployment;
* flexible usage;
* competitive prices;
* suitable for temporary workloads;
* easy access to high-end GPUs.

### Disadvantages

* less enterprise-oriented;
* potentially less predictable availability;
* fewer integrated cloud services;
* not necessarily designed to become the company's complete production infrastructure.

For the current 10,000-document annotation stage, these platforms are highly suitable because the GPU does not need to operate continuously.

# 5. Why AWS/Azure Can Be More Expensive

The higher price of AWS or Azure does not mean that their GPUs are necessarily more expensive to operate physically.

The difference comes from the service being purchased.

A hyperscaler provides:

```text
GPU + CPU / RAM + Networking + Storage + Virtualization + Security + Identity management + Monitoring + Availability infrastructure + Enterprise services
```

A specialized GPU provider can focus much more narrowly on:

```text
GPU + CPU / RAM + Basic infrastructure
```

Therefore, a specialized provider can offer the GPU at a lower price.

The additional infrastructure provided by AWS or Azure is valuable when the company needs it, but it does not necessarily provide enough additional value to justify the additional cost for a temporary batch annotation workload.

The current workload is particularly well suited to specialized GPU providers because the GPU can be rented only for the period required to annotate the dataset.

# 6. Current H100 80 GB Price Comparison

The following comparison focuses on **80 GB H100-class GPUs or larger**, excluding smaller GPUs that are not appropriate for the planned teacher-model experiments.

<table>
  <thead>
    <tr>
      <th>Provider</th>
      <th>Category</th>
      <th>GPU</th>
      <th>VRAM</th>
      <th>On-demand price</th>
      <th>Cost / 100 h</th>
      <th>Cost / 500 h</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>RunPod</strong></td>
      <td>Developer GPU cloud</td>
      <td>H100 PCIe</td>
      <td>80 GB</td>
      <td><strong>$2.89/h</strong></td>
      <td>$289</td>
      <td>$1,445</td>
    </tr>
    <tr>
      <td><strong>Scaleway</strong></td>
      <td>AI cloud</td>
      <td>H100 PCIe</td>
      <td>80 GB</td>
      <td><strong>from €2.86/h</strong></td>
      <td>€286</td>
      <td>€1,430</td>
    </tr>
    <tr>
      <td><strong>Verda</strong></td>
      <td>AI cloud</td>
      <td>H100 SXM5</td>
      <td>80 GB</td>
      <td><strong>$3.25/h</strong></td>
      <td>$325</td>
      <td>$1,625</td>
    </tr>
    <tr>
      <td><strong>Nebius</strong></td>
      <td>AI cloud</td>
      <td>H100</td>
      <td>80 GB</td>
      <td><strong>$3.85/h</strong></td>
      <td>$385</td>
      <td>$1,925</td>
    </tr>
    <tr>
      <td><strong>Lambda</strong></td>
      <td>AI cloud</td>
      <td>H100 SXM</td>
      <td>80 GB</td>
      <td><strong>$3.99/h</strong></td>
      <td>$399</td>
      <td>$1,995</td>
    </tr>
    <tr>
      <td><strong>DigitalOcean</strong></td>
      <td>Developer / cloud</td>
      <td>H100</td>
      <td>80 GB</td>
      <td><strong>$4.41/h</strong></td>
      <td>$441</td>
      <td>$2,205</td>
    </tr>
    <tr>
      <td><strong>Paperspace</strong></td>
      <td>Developer GPU cloud</td>
      <td>H100</td>
      <td>80 GB</td>
      <td><strong>$5.95/h</strong></td>
      <td>$595</td>
      <td>$2,975</td>
    </tr>
    <tr>
      <td><strong>AWS</strong></td>
      <td>Hyperscaler</td>
      <td>H100</td>
      <td>80 GB</td>
      <td><strong>~$5.19/h</strong></td>
      <td>~$519</td>
      <td>~$2,595</td>
    </tr>
    <tr>
      <td><strong>Google Cloud</strong></td>
      <td>Hyperscaler</td>
      <td>H100</td>
      <td>80 GB</td>
      <td><strong>~$11/h</strong></td>
      <td>~$1,100</td>
      <td>~$5,500</td>
    </tr>
  </tbody>
</table>

<p><em>Prices are approximate on-demand rates and may vary depending on region, configuration, and provider pricing.</em></p>

### Provider sources

**RunPod:** H100 PCIe 80 GB is currently listed at $2.89/hour on Secure Cloud. [RunPod GPU pricing](https://www.runpod.io/pricing?utm_source=chatgpt.com)

**Scaleway:** H100 PCIe instances are listed from €2.86/GPU-hour. [Scaleway GPU pricing](https://www.scaleway.com/en/pricing/gpu/?utm_source=chatgpt.com)

**Verda:** H100 SXM5 80 GB is currently listed at $3.25/hour on-demand and $1.63/hour spot. [Verda GPU pricing](https://verda.com/pricing?utm_source=chatgpt.com)

**Nebius:** H100 is currently listed at $3.85/hour on-demand, with lower preemptible pricing available. [Nebius GPU pricing](https://nebius.com/prices?utm_source=chatgpt.com)

**Lambda:** H100 80 GB instances are listed at approximately $3.99/GPU-hour. [Lambda GPU instances](https://lambda.ai/instances?utm_source=chatgpt.com)

**DigitalOcean:** H100 GPU Droplets are currently listed at $4.41/hour. [DigitalOcean GPU pricing](https://www.digitalocean.com/pricing/gpu-droplets?utm_source=chatgpt.com)

**Paperspace:** H100 is currently listed at $5.95/hour. [Paperspace pricing](https://www.paperspace.com/pricing?utm_source=chatgpt.com)

**AWS:** The P5 family uses H100 80 GB GPUs. AWS currently publishes H100 Capacity Block pricing around $5.19/GPU-hour in applicable regions. [AWS P5 instances](https://aws.amazon.com/ec2/instance-types/p5/?utm_source=chatgpt.com)

# 7. Spot / Preemptible Compute

The annotation workload is particularly suitable for **spot or preemptible GPUs**. The objective is to process a finite dataset:

> 10,000 documents

rather than maintain a GPU continuously.


If the workload is designed to resume from checkpoints, temporary interruptions are acceptable.

This makes spot/preemptible pricing potentially much cheaper than standard on-demand pricing.

For example, Verda currently lists:

* H100 SXM5 80 GB on-demand: **$3.25/hour**


* H100 SXM5 80 GB spot: **$1.63/hour**.

Nebius currently lists H100:

* on-demand: **$3.85/hour**


* preemptible: **$2.15/hour**.

However, spot/preemptible instances can be interrupted, so they should only be used if the annotation pipeline supports checkpointing and restarting.

# 9. Recommended Approach

For the current stage, the infrastructure requirement is temporary GPU compute for generating the 10,000-document training dataset.

A permanent GPU server is therefore unnecessary.

The recommended process is:

### Step 1 — Evaluate larger teacher models

Use an H100 80 GB or equivalent GPU to benchmark large VLMs against the existing Qwen2.5-VL-7B results.

### Step 2 — Select the teacher

Select the model that provides the best balance between:

* annotation accuracy;
* inference speed;
* GPU memory usage;
* cost.

### Step 3 — Annotate the FATURA dataset

Run the selected teacher over the 10,000 original document images and generate annotations directly in the desired target structure.

### Step 4 — Validate the generated dataset

Perform quality checks and correct a representative subset of the annotations.

### Step 5 — Fine-tune a smaller VLM

Use the resulting dataset to fine-tune a smaller model specifically for the company's document types and extraction schema.

### Step 6 — Optimize production infrastructure

Once the smaller production model is known, the compute requirements will be significantly lower and a separate infrastructure analysis can be performed for production deployment.



# 10. Final Conclusion

The current compute requirement is driven by the **teacher model used to generate the training dataset**, not by the final production model.

The initial Qwen2.5-VL-7B experiment on 200 random documents did not provide sufficiently reliable annotations. A larger VLM must therefore be evaluated as the teacher.

Because a 32B-class model requires approximately 64 GB just for its BF16/FP16 weights, an **80 GB GPU is an appropriate baseline** for this stage.

The 10,000 FATURA documents should be processed directly from their original images because the existing FATURA annotations do not contain all of the information required by the target schema. Schema conversion alone would therefore not be sufficient.

For this finite batch workload, there is no requirement to maintain a GPU continuously. A rented H100 can be started when annotation is required and terminated afterward. Spot/preemptible compute can reduce the cost further if the pipeline supports checkpointing.

## Estimated Cost and Annotation Runtime

The exact cost of annotating the 10,000 FATURA documents cannot be determined in advance with complete precision.

The GPU rental price is only one component of the total cost. The actual runtime depends on several factors that have not yet been benchmarked with the final teacher model:

* the exact VLM selected;
* image resolution;
* number of pages per document;
* prompt length;
* size of the generated JSON response;
* batching strategy;
* quantization;
* inference framework;
* model loading time;
* GPU availability;
* setup and environment installation time;
* potential interruptions when using spot/preemptible instances.

For example, the same H100 can have significantly different effective costs depending on whether the model processes documents sequentially or in batches.

For planning purposes, an average reference price of approximately $3.5/hour per H100 can therefore be used for the initial estimation.

### Estimated Runtime

The runtime for processing 10,000 documents with Qwen2.5-VL-32B cannot be reliably predicted from the model size alone. A benchmark on the actual documents is required.

A reasonable initial planning range is approximately:

| Scenario     | Average processing time / document | 10,000 documents |
| ------------ | ---------------------------------: | ---------------: |
| Fast         |                    ~5 sec/document |        ~14 hours |
| Medium       |                   ~10 sec/document |        ~28 hours |
| Conservative |                   ~20 sec/document |        ~56 hours |
| Slow         |                   ~30 sec/document |        ~83 hours |

These figures are **planning scenarios, not measured Qwen2.5-VL-32B performance**. They should therefore not be presented as guaranteed inference times.

Using an estimated H100 price of **$3.5/hour**, the corresponding GPU costs would be approximately:

<table>
  <thead>
    <tr>
      <th>Runtime</th>
      <th>Estimated H100 cost</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>14 hours</td>
      <td>~$49</td>
    </tr>
    <tr>
      <td>28 hours</td>
      <td>~$98</td>
    </tr>
    <tr>
      <td>56 hours</td>
      <td>~$196</td>
    </tr>
    <tr>
      <td>83 hours</td>
      <td>~$291</td>
    </tr>
  </tbody>
</table>

Additional time should be allowed for:

* environment setup;
* downloading the model;
* downloading/uploading the dataset;
* initial testing;
* failed jobs;
* checkpointing;
* possible GPU interruptions.

Consequently, a realistic initial budget of **approximately 50–250 $ for the GPU compute required to annotate 10,000 documents** can be used as a planning estimate, assuming an H100-class GPU and successful batch processing.

This should not be interpreted as the final cost. The exact figure will only be known after conducting a throughput benchmark using the selected teacher model and a representative subset of the FATURA documents.

The recommended procedure is therefore to first process a small benchmark set, for example **100–200 documents**, and measure:

$$
\text{documents/hour}
$$

Then the total annotation cost can be calculated directly:

$$
\text{Total Cost}
=
\frac{10,000}{\text{documents/hour}}
\times
\text{GPU price/hour}
$$

This provides a much more reliable estimate than attempting to predict the complete 10,000-document runtime from the model size alone.